In [12]:
!jupyter nbconvert --to script 04_evaluation_and_reporting.ipynb


[NbConvertApp] Converting notebook 04_evaluation_and_reporting.ipynb to script
[NbConvertApp] Writing 4683 bytes to 04_evaluation_and_reporting.py


In [18]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.estimator import Estimator
import sagemaker
import boto3
import os

# === SETUP ===
role = sagemaker.get_execution_role()
session = boto3.Session()
region = session.region_name
pipeline_session = PipelineSession()

# === PARAMETERS ===
input_data = ParameterString(name="InputData", default_value="s3://your-bucket/raw_data/")
baseline_threshold = ParameterFloat(name="ModelAccuracyThreshold", default_value=0.8)

# === STEP 1: Data and Featurestore ===
data_processor = SKLearnProcessor(
    framework_version="0.23-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="data-prep",
    sagemaker_session=pipeline_session
)

step_process_data = ProcessingStep(
    name="ProcessData",
    processor=data_processor,
    code="01_data_and_featurestore.py"
)

# === STEP 2: Feature Engineering ===
step_feature_engineering = ProcessingStep(
    name="FeatureEngineering",
    processor=data_processor,
    code="02_feature_engineering.py"
)

# === STEP 3: Model Training ===
estimator = Estimator(
    image_uri="382416733822.dkr.ecr.us-east-1.amazonaws.com/xgboost:1.5-1",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path="s3://your-bucket/model_artifacts/",
    sagemaker_session=pipeline_session
)

step_train = TrainingStep(
    name="ModelTraining",
    estimator=estimator,
    inputs={"train": "s3://your-bucket/train_data/train.csv"}
)

# === STEP 4: Model Evaluation ===
sklearn_processor = SKLearnProcessor(
    framework_version="0.23-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    sagemaker_session=pipeline_session
)

evaluation_report = PropertyFile(
    name="evaluation",
    output_name="output",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="ModelEvaluation",
    processor=sklearn_processor,
    code="04_evaluation_and_reporting.py",
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="output",
            source="/opt/ml/processing/output"
        )
    ],
    property_files=[evaluation_report]
)

# === STEP 5: Register Model ===
model_name = "cicd-pipeline-model"

model = Model(
    image_uri=estimator.image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    name=model_name
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=step_eval.properties.ProcessingOutputConfig.Outputs["output"].S3Output.S3Uri,
        content_type="application/json"
    )
)

step_register = RegisterModel(
    name="RegisterModel",
    estimator=estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="YourModelPackageGroupName",
    model_metrics=model_metrics,
    model=model
)

# === STEP 6: Batch Transform ===
transformer = Transformer(
    model_name=model_name,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path="s3://your-bucket/batch_output/",
    sagemaker_session=pipeline_session
)

step_batch = TransformStep(
    name="BatchInference",
    transformer=transformer,
    inputs=TransformInput(
        data="s3://your-bucket/batch_input/",
        content_type="text/csv"
    )
)

# === STEP 7: CI/CD Placeholder (external triggers)
# No internal logic needed here. CI/CD logic handled externally via CodePipeline, etc.

# === BUILD PIPELINE ===
pipeline = Pipeline(
    name="CI-CD-Train-Eval-Register-Infer",
    parameters=[input_data, baseline_threshold],
    steps=[
        step_process_data,
        step_feature_engineering,
        step_train,
        step_eval,
        step_register,
        step_batch
    ],
    sagemaker_session=pipeline_session
)

# === DEPLOY PIPELINE ===
response = pipeline.upsert(role_arn=role)
print("✅ Pipeline created or updated.")
print("Pipeline ARN:", response['PipelineArn'])


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


✅ Pipeline created or updated.
Pipeline ARN: arn:aws:sagemaker:us-east-1:277277490522:pipeline/CI-CD-Train-Eval-Register-Infer
